In [ ]:
# Kaggle runtime inspection
import os, json

print('Current working dir:', os.getcwd())
print('Kaggle input dirs:')
for root, dirs, files in os.walk('/kaggle/input'):
    print('  ', root)
    break


In [ ]:
# ===== Cell 1 =====
# Clone the repo (if needed) and checkout the DiCMA feature branch
REPO_DIR = '/kaggle/working/CLIP-ReID'

# Clone if missing
if not os.path.exists(REPO_DIR):
    ! git clone https://github.com/Debjit-Dhar/CLIP-ReID.git $REPO_DIR

# Move into repo and checkout branch
%cd $REPO_DIR
! git fetch --all
! git checkout feat/dicma-w2
! git pull --ff-only

# Install required dependencies
! pip install -q yacs timm scikit-image tqdm ftfy regex
! pip install -q torch torchvision

# Print git branch to confirm
! git branch


In [ ]:
# ===== Cell 2 =====
# Inspect input dataset structure (Market-1501 expected under /kaggle/input)
DATA_ROOT = '/kaggle/input/market-1501'
! ls -la $DATA_ROOT | head

# Ensure output directory exists
OUTPUT_DIR = '/kaggle/working/output'
! mkdir -p $OUTPUT_DIR

print('DATA_ROOT:', DATA_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)


In [ ]:
# ===== Cell 3 =====
# Run training using DiCMA (Market-1501)
# NOTE: this is a minimal sanity run (1 epoch). Increase MAX_EPOCHS for full training.
! python /kaggle/working/train.py \
    --config_file configs/person/vit_dicma.yml \
    --use_dicma \
    --dicma_alpha 1.0 \
    --dicma_beta 0.1 \
    SOLVER.MAX_EPOCHS 1 \
    SOLVER.EVAL_PERIOD 1 \
    DATASETS.ROOT_DIR $DATA_ROOT \
    OUTPUT_DIR $OUTPUT_DIR


In [ ]:
# ===== Cell 4 =====
# Evaluate the trained model on Market-1501

import glob, os
ckpts = glob.glob(os.path.join(OUTPUT_DIR, '*_1.pth'))
print('Checkpoints found:', ckpts)

if ckpts:
    ! python test.py --config_file configs/person/vit_dicma.yml TEST.WEIGHT {ckpts[0]} DATASETS.ROOT_DIR $DATA_ROOT
else:
    print('No checkpoint found. Ensure training completed successfully.')


In [ ]:
# ===== Cell 5 =====
# Inspect DiCMA training logs (W2 / covariance / GW statistics) if available
LOGS = os.path.join(OUTPUT_DIR, 'log.txt')
if os.path.exists(LOGS):
    ! tail -n 50 $LOGS
else:
    print('No log file found at', LOGS)
